# Free-Space Optical Propagation

The paraxial wave equation has the same mathematical form as the
free-particle Schrödinger equation after replacing time with propagation
distance.

This notebook compares split-step propagation of a Gaussian optical field
with ballistic propagation of Monte Carlo phase-space particles.

## Before running

From the repository's top-level folder:

```bash
python3 -m pip install -e .
```


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from mcw import (
    gaussian_wavepacket,
    split_operator_propagate,
    sample_wigner_gaussian,
    gaussian_kde_marginal,
)
from mcw.plots import compare_densities

from mcw import propagate_free


In [ ]:
distance_step = 0.05
nsteps = 400
steps = {0, nsteps // 4, nsteps // 2, nsteps}

x = np.linspace(-40.0, 40.0, 2**12, endpoint=False)
field0 = gaussian_wavepacket(x, sigma=2.0)


In [ ]:
field_snapshots = split_operator_propagate(
    field0,
    x,
    distance_step,
    nsteps,
    lambda grid: np.zeros_like(grid),
    snapshot_steps=steps,
)

xs0, ps0, signs = sample_wigner_gaussian(
    20_000,
    sigma=2.0,
    seed=123,
)

ray_snapshots = propagate_free(
    xs0,
    ps0,
    distance_step,
    nsteps,
    snapshot_steps=steps,
)


In [ ]:
for step in sorted(steps):
    reference = np.abs(field_snapshots[step]) ** 2
    reference /= np.trapz(reference, x)

    xs, _ = ray_snapshots[step]
    estimate = gaussian_kde_marginal(
        xs,
        signs,
        x,
        bandwidth=0.25,
    )

    compare_densities(
        x,
        reference,
        estimate,
        title=(
            "Free paraxial propagation, "
            f"z={step * distance_step:.2f}"
        ),
    )
    plt.show()
